# MEME: ED disposition from clinical pseudo-notes in PyHealth

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sunlabuiuc/PyHealth/blob/master/examples/meme_mimic4_ed_disposition.ipynb)

This notebook runs, at a small scale, the main experiment of

> Simon A. Lee, Sujay Jain, Alex Chen, Kyoka Ono, Arabdha Biswas, Ákos Rudas, Jennifer Fang, Jeffrey N. Chiang. **Clinical decision support using pseudo-notes from multiple streams of EHR data.** *npj Digital Medicine* 8, 394 (2025). [doi:10.1038/s41746-025-01777-x](https://doi.org/10.1038/s41746-025-01777-x) · preprint: [arXiv:2402.00160](https://arxiv.org/abs/2402.00160) ("Emergency Department Decision Support using Clinical Pseudo-notes") · [code](https://github.com/Simonlee711/MEME) (MIT)

**The method.** MEME (Multiple Embedding Model for EHR) writes each domain of an emergency department (ED) visit as a short templated text, a "pseudo-note": arrival, triage, medication reconciliation, vital signs, medications dispensed (Pyxis) and ED diagnosis codes. A frozen clinical BERT (MedBERT) embeds each note, self-attention runs across the note embeddings, and a linear head predicts the ED disposition: admitted or discharged home.

**In PyHealth** this is three pieces:

- `MIMIC4Dataset(ehr_root=..., ed_root=..., ed_tables=[...])` (or `MIMIC4EDDataset`) loads MIMIC-IV-ED next to the hospital tables.
- `EDDispositionMIMIC4` writes the pseudo-notes, one text field per domain; `EDOutcomesMIMIC4` is the paper's multitask setting (discharge location, in-hospital mortality, ICU) for admitted patients.
- `MEME` is the model: frozen encoder, embedding cache, attention across domains, linear head.

**Leakage.** ED diagnosis codes are assigned at the end of the stay, so they partly encode the disposition. The paper uses them; PyHealth leaves them out unless `include_codes=True`. Section 5 runs both.

## 0. Install

In [1]:
# Installs PyHealth from GitHub. MEME is in PyHealth once the pull request
# that adds it is merged into master; until then this install does not
# include it.
import importlib.util
import subprocess
import sys

if "google.colab" in sys.modules or importlib.util.find_spec("pyhealth") is None:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/sunlabuiuc/PyHealth"],
        check=True,
    )

## 1. Settings

In [2]:
import os

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("TQDM_DISABLE", "1")  # no progress bars

# ---- switch: "demo" (open MIMIC-IV-ED demo, ~100 patients) or "full" (credentialed) ----
DATA = "demo"
MIMIC4_ROOT = "/content/mimic-iv/2.2"        # folder with hosp/ and icu/, for DATA="full"
MIMIC4_ED_ROOT = "/content/mimic-iv-ed/2.2"  # folder with ed/, for DATA="full"

MODEL_NAME = "Charangan/MedBERT"  # the paper's encoder
MAX_LENGTH = 512
CACHE_DIR = "./pyhealth_cache"
SEED = 0

## 2. Data: the open MIMIC-IV-ED and MIMIC-IV demos

Both demos are open on PhysioNet (ODbL). The ED demo has 222 ED stays of 64 of the 100 MIMIC-IV demo patients; the MIMIC-IV demo supplies age (`patients`) and, for the multitask labels, `admissions` and `icustays`. About 1 MB is downloaded.

In [3]:
import urllib.request

ED_URL = "https://physionet.org/files/mimic-iv-ed-demo/2.2/"
EHR_URL = "https://physionet.org/files/mimic-iv-demo/2.2/"
ED_TABLES = ["edstays", "triage", "vitalsign", "medrecon", "pyxis", "diagnosis"]
EHR_TABLES = ["hosp/patients", "hosp/admissions", "icu/icustays"]

if DATA == "demo":
    ED_ROOT, EHR_ROOT = "./mimic-iv-ed-demo/2.2", "./mimic-iv-demo/2.2"
    files = [(ED_URL, ED_ROOT, f"ed/{t}") for t in ED_TABLES]
    files += [(EHR_URL, EHR_ROOT, t) for t in EHR_TABLES]
    for url, root, table in files:
        path = os.path.join(root, table + ".csv.gz")
        if not os.path.exists(path):
            os.makedirs(os.path.dirname(path), exist_ok=True)
            urllib.request.urlretrieve(url + table + ".csv.gz", path)
    print("downloaded to", ED_ROOT, "and", EHR_ROOT)
else:
    ED_ROOT, EHR_ROOT = MIMIC4_ED_ROOT, MIMIC4_ROOT

downloaded to ./mimic-iv-ed-demo/2.2 and ./mimic-iv-demo/2.2


## 3. Pseudo-notes

`EDDispositionMIMIC4` makes one sample per ED stay that ends at home (label 0) or in admission or death (label 1), as in the paper's code; transfers and patients who left are left out. With `include_codes=True` it adds the `codes` note. Below is one sample.

In [4]:
import logging

from pyhealth.datasets import MIMIC4Dataset
from pyhealth.tasks import EDDispositionMIMIC4

logging.getLogger("pyhealth").setLevel(logging.WARNING)  # quieter loading

dataset = MIMIC4Dataset(
    ehr_root=EHR_ROOT,
    ed_root=ED_ROOT,
    ed_tables=["triage", "vitalsign", "medrecon", "pyxis", "diagnosis"],
    cache_dir=CACHE_DIR,
)
# with the codes note, so both input sets come from one run; the leakage-safe
# model below simply does not read it
samples = dataset.set_task(EDDispositionMIMIC4(include_codes=True), num_workers=1)
labels = [int(s["disposition"]) for s in samples]
print(f"{len(samples)} ED stays, {sum(labels)} admitted ({sum(labels) / len(labels):.0%})\n")
example = samples[0]
for domain in ["arrival", "triage", "medrecon", "vitals", "pyxis", "codes"]:
    text = example[domain]
    print(f"[{domain}] {text[:300]}{' ...' if len(text) > 300 else ''}\n")

210 ED stays, 150 admitted (71%)

[arrival] A 52 year old white female, arrived via ambulance at 2180-05-06 19:17:00.

[triage] At triage: temperature was 98.4, pulse was 70, respirations was 16, o2 saturation was 97, systolic blood pressure was 106, diastolic blood pressure was 63, pain was 0, chief complaint was abd pain, abdominal distention. Acuity score was 3.

[medrecon] The patient was previously taking the following medications: ipratropium bromide [atrovent hfa], asthma/copd - anticholinergic agents, inhaled short acting. raltegravir [isentress], antiretroviral - hiv-1 integrase strand transfer inhibitors. spironolactone [aldactone], aldosterone receptor antagoni ...

[vitals] The patient had the following vitals: At 2180-05-06 23:04:00, temperature was 97.7, pulse was 79, respirations was 16, o2 saturation was 98, systolic blood pressure was 107, diastolic blood pressure was 60, pain was 0.

[pyxis] The patient did not receive any medications.

[codes] The patient received th

## 4. Embed the notes with frozen MedBERT

`MEME` embeds every note once (the `[CLS]` token of MedBERT) and caches the vectors in `CACHE_DIR`, so the training epochs, the repeated splits and reruns reuse them. On a CPU, embedding the demo's ~1,200 notes takes a few minutes.

In [5]:
import numpy as np
import torch

from pyhealth.models import MEME

DOMAINS = ["arrival", "triage", "medrecon", "vitals", "pyxis"]  # no codes
gpu = torch.cuda.is_available()
encoder = MEME(
    samples, feature_keys=DOMAINS, model_name=MODEL_NAME, max_length=MAX_LENGTH,
    batch_size=32 if gpu else 8, cache_dir=os.path.join(CACHE_DIR, "meme"),
)
if gpu:
    encoder.to("cuda")
# embed every note up front, codes included
_ = encoder.embed([s[d] for s in samples for d in DOMAINS + ["codes"]])
print("embedding size", encoder.embedding_dim)

embedding size 768


## 5. Train the attention head, without and with the diagnosis codes

`MEME` is a regular PyHealth model, so `Trainer` fits it. Only the attention head and the linear layer train; checkpoints hold only those. The models below get their embeddings from the cached encoder above through `embed_fn`, so MedBERT is not reloaded. The second input set adds the ED diagnosis codes, as in the paper.

The paper splits ED visits 70/15/15. Here the split is by patient, so no patient is in two splits. On the demo one test split has only about 50 stays from about 20 patients, so the results change a lot from split to split; the cell repeats the experiment over 10 patient splits and reports the mean and standard deviation.

As a check on the embeddings, the cell also fits a strongly regularized logistic regression on the concatenated note embeddings (no attention). MEME's head has about 2.4M parameters and the demo has ~105 training stays, so the head is trained gently (learning rate 1e-4, weight decay 0.01, 20 epochs). In a check outside this notebook (not reproduced by its cells), the paper code's learning rate of 1e-3 and 30 epochs fit the training split almost perfectly and scored below chance on the test split. That is overfitting at this size and says nothing about the full data.

In [6]:
from pyhealth.datasets import get_dataloader, split_by_patient
from pyhealth.trainer import Trainer

logging.getLogger("pyhealth.trainer").setLevel(logging.WARNING)
METRICS = ["roc_auc", "pr_auc", "f1"]
EPOCHS = 20 if DATA == "demo" else 10
SPLIT_SEEDS = range(10) if DATA == "demo" else [SEED]
ratios = [0.5, 0.2, 0.3] if DATA == "demo" else [0.7, 0.15, 0.15]


from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score


def logistic(domains, train, test):
    def xy(part):
        rows = [part[i] for i in range(len(part))]
        x = torch.cat([encoder.embed([r[d] for r in rows]) for d in domains], 1)
        return x.cpu().numpy(), np.array([int(r["disposition"]) for r in rows])

    (x_train, y_train), (x_test, y_test) = xy(train), xy(test)
    p = LogisticRegression(C=0.01, max_iter=5000).fit(x_train, y_train).predict_proba(x_test)[:, 1]
    return {"roc_auc": roc_auc_score(y_test, p), "pr_auc": average_precision_score(y_test, p),
            "f1": f1_score(y_test, p >= 0.5)}


def fit(domains, train, val, test, seed):
    torch.manual_seed(seed)
    model = MEME(samples, feature_keys=domains, embed_fn=encoder.embed)
    trainer = Trainer(model=model, metrics=METRICS, enable_logging=False)
    trainer.train(
        train_dataloader=get_dataloader(train, batch_size=32, shuffle=True),
        val_dataloader=get_dataloader(val, batch_size=64),
        epochs=EPOCHS, monitor="roc_auc", optimizer_params={"lr": 1e-4},
        weight_decay=1e-2,
    )
    return trainer.evaluate(get_dataloader(test, batch_size=64))


INPUTS = {"MEME, no codes": DOMAINS,  # PyHealth's default
          "MEME, with codes": DOMAINS + ["codes"]}  # the paper's inputs
runs = {name: [] for name in INPUTS}
runs.update({f"LR on embeddings, {k.split(', ')[1]}": [] for k in INPUTS})
for seed in SPLIT_SEEDS:
    train, val, test = split_by_patient(samples, ratios, seed=seed)
    n_pos = sum(int(test[i]["disposition"]) for i in range(len(test)))
    line = f"split {seed}: test {len(test)} stays, {n_pos} admitted |"
    for name, domains in INPUTS.items():
        runs[name].append(fit(domains, train, val, test, seed))
        runs[f"LR on embeddings, {name.split(', ')[1]}"].append(
            logistic(domains, train, test))
        line += f" {name.split(', ')[1]}: AUROC {runs[name][-1]['roc_auc']:.3f}"
    print(line)

split 0: test 51 stays, 38 admitted | no codes: AUROC 0.478 with codes: AUROC 0.609


split 1: test 54 stays, 35 admitted | no codes: AUROC 0.614 with codes: AUROC 0.708


split 2: test 50 stays, 34 admitted | no codes: AUROC 0.610 with codes: AUROC 0.568


split 3: test 41 stays, 35 admitted | no codes: AUROC 0.410 with codes: AUROC 0.395


split 4: test 62 stays, 47 admitted | no codes: AUROC 0.445 with codes: AUROC 0.597


split 5: test 42 stays, 31 admitted | no codes: AUROC 0.443 with codes: AUROC 0.651


split 6: test 73 stays, 54 admitted | no codes: AUROC 0.636 with codes: AUROC 0.752


split 7: test 54 stays, 37 admitted | no codes: AUROC 0.580 with codes: AUROC 0.700


split 8: test 94 stays, 66 admitted | no codes: AUROC 0.457 with codes: AUROC 0.669


split 9: test 63 stays, 48 admitted | no codes: AUROC 0.381 with codes: AUROC 0.629


## 6. The paper's numbers next to this run

The paper's numbers (its Tables 2 to 4) are for ED disposition on the full MIMIC-IV-ED (400,019 visits, split 70/15/15 by visit). **The demo run is a smoke test:** it checks that the pipeline runs end to end on real MIMIC-IV-ED tables. With ~105 training stays and ~50 test stays per split it cannot test the paper's claim, and its numbers mostly reflect the small sample. In the recorded run, MEME without the codes is at chance on the demo (mean AUROC 0.51 ± 0.09 over 10 splits), below the logistic regression on the same embeddings (0.58). The data does carry signal at this size: in a check outside this notebook (not reproduced by its cells), over 50 patient splits, triage acuity alone gave AUROC 0.69. So the 2.4M-parameter head cannot learn from ~105 training stays; this is not a sign of a broken pipeline. With the diagnosis codes, MEME reaches 0.63 and the logistic regression 0.67. Adding the codes raises every metric, the direction the leakage concern predicts, but the demo is too small to measure it. To compare with the paper, set `DATA = "full"` (section 7).

In [7]:
import pandas as pd

rows = [
    {"model": "MEME (paper, MIMIC-IV-ED, with codes)", "AUROC": "0.991", "AUPRC": "0.983", "F1": "0.943"},
    {"model": "MC-BEC (paper's strongest baseline)", "AUROC": "0.968", "AUPRC": "0.935", "F1": "0.912"},
]
for name, results in runs.items():
    row = {"model": f"{name} [{DATA}, {len(results)} split(s)]"}
    for metric, column in [("roc_auc", "AUROC"), ("pr_auc", "AUPRC"), ("f1", "F1")]:
        values = np.array([r[metric] for r in results])
        row[column] = f"{values.mean():.3f} ± {values.std():.3f}" if len(values) > 1 else f"{values[0]:.3f}"
    rows.append(row)
pd.DataFrame(rows).set_index("model")

,AUROC,AUPRC,F1
model,,,
"MEME (paper, MIMIC-IV-ED, with codes)",0.991,0.983,0.943
MC-BEC (paper's strongest baseline),0.968,0.935,0.912
"MEME, no codes [demo, 10 split(s)]",0.505 ± 0.090,0.765 ± 0.054,0.827 ± 0.031
"MEME, with codes [demo, 10 split(s)]",0.628 ± 0.094,0.817 ± 0.045,0.839 ± 0.039
"LR on embeddings, no codes [demo, 10 split(s)]",0.582 ± 0.070,0.796 ± 0.042,0.826 ± 0.026
"LR on embeddings, with codes [demo, 10 split(s)]",0.671 ± 0.061,0.845 ± 0.034,0.838 ± 0.025


## 7. Full MIMIC-IV-ED (credentialed)

With PhysioNet access to [MIMIC-IV](https://physionet.org/content/mimiciv/) and [MIMIC-IV-ED](https://physionet.org/content/mimic-iv-ed/), mount the data, set `DATA = "full"`, `MIMIC4_ROOT` and `MIMIC4_ED_ROOT` in section 1, and rerun sections 3 to 6. Use a GPU runtime: about 400,000 stays x 6 notes is a few GPU hours of MedBERT inference, once; the cache makes later runs fast. For a quick check, pass `dev=True` to `MIMIC4Dataset` first.

**The multitask setting.** Swap the task for `EDOutcomesMIMIC4(include_codes=...)`: for admitted patients it predicts, together, discharge to a place other than home, in-hospital mortality and an ICU stay (a multilabel target). `MEME` handles it unchanged.

**What differs from the paper's code.**

- The reference model attends over the token states of all notes concatenated (up to 6 x 512 tokens). `MEME` attends over one `[CLS]` vector per note, as the paper describes, which allows caching the embeddings.
- The reference split is by stay; here it is by patient.
- The reference preprocessing adds insurance, language and marital status from the hospital `admissions` table to the arrival note. That table only has rows for admitted patients, so those fields reveal the label; PyHealth never uses them. The reference trainer also replaces every medication-reconciliation note with a constant text, while PyHealth writes the real one.
- Vital signs and Pyxis records cover the whole ED stay, and longer stays are more often admitted. `EDDispositionMIMIC4(observation_hours=...)` keeps only the first hours after arrival, for a prediction made during the visit.